In [1]:
!ls /kaggle/input/datasets/kiryall

safer-code  safer-dataset-2


In [2]:
!cp -r /kaggle/input/datasets/kiryall/safer-code /kaggle/working/project

In [3]:
%cd /kaggle/working/project

/kaggle/working/project


In [4]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.5 MB/s eta 0:00:0000:01


In [5]:
import yaml

DATASET_PATH = "/kaggle/input/datasets/kiryall/safer-dataset-2/"

# читаем оригинальный yaml
with open(f"{DATASET_PATH}/data.yaml", "r") as f:
    data = yaml.safe_load(f)

# переписываем пути
data["train"] = f"{DATASET_PATH}/images/train"
data["val"] = f"{DATASET_PATH}/images/val"

# сохраняем временный yaml
TEMP_DATA_YAML = "/kaggle/working/data.yaml"

with open(TEMP_DATA_YAML, "w") as f:
    yaml.dump(data, f)

print("NEW DATA YAML:", TEMP_DATA_YAML)

NEW DATA YAML: /kaggle/working/data.yaml


In [7]:
!python -m scripts.train_yolo \
  --config /kaggle/working/project/configs/model_config.yaml \
  --data /kaggle/working/data.yaml

2026-04-08 20:58:06,585 - INFO - Loaded YOLO model from models/yolo/yolo26s-obb.pt
2026-04-08 20:58:06,585 - INFO - Starting training with experiment name: plate_detector_20260408_2058
2026-04-08 20:58:06,585 - INFO - Using data: /kaggle/working/data.yaml
2026-04-08 20:58:06,585 - INFO - Config: /kaggle/working/project/configs/model_config.yaml
2026-04-08 20:58:06,585 - INFO - Device: 0
Ultralytics 8.4.36 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, fre

In [8]:
import pandas as pd
from pathlib import Path

RUNS_DIR = Path("runs/seg")

# ищем ВСЕ experiments (вложенные папки)
experiments = list(RUNS_DIR.glob("*/*"))

results = []

for exp in experiments:
    results_file = exp / "results.csv"
    
    if results_file.exists():
        df = pd.read_csv(results_file)
        best = df.loc[df['metrics/mAP50(B)'].idxmax()]
        
        results.append({
            "experiment": exp.name,
            "type": exp.parent.name,
            "mAP50": best['metrics/mAP50(B)'],
            "mAP50-95": best['metrics/mAP50-95(B)'],
            "precision": best['metrics/precision(B)'],
            "recall": best['metrics/recall(B)'],
        })

comparison = pd.DataFrame(results)

print("\n===== FINAL COMPARISON =====")
print(comparison)


===== FINAL COMPARISON =====
                     experiment type    mAP50  mAP50-95  precision   recall
0  plate_detector_20260408_2058  aug  0.98371   0.88749    0.98006  0.94286
